# Parser di documenti per knowledge base

Progetto del corso *Programmazione con Python* — Master ProfessionAI.

Programma Python che, data una cartella di documenti in formati diversi (`.txt`, `.md`, `.csv`), ne estrae contenuto e metadati, li normalizza in una rappresentazione comune e costruisce un indice della knowledge base, salvato in JSON e interrogabile da riga di comando.

**Repository:** [https://github.com/massimiliano-cantore/kb-parser](https://github.com/massimiliano-cantore/kb-parser) — codice, moduli, archivio di prova e README con le istruzioni.

Il notebook è diviso in tre parti:

- **Parte A** — uso del programma da riga di comando sull'archivio di prova;
- **Parte B** — uso delle classi da Python: ricaricamento dell'indice, ricerche, estensione con un nuovo formato;
- **Parte C** — prova di robustezza su file problematici creati al momento, che non fanno parte dell'archivio di prova.

Per eseguirlo: *Runtime → Esegui tutte*.

## Installazione

Clonazione del repository e installazione del pacchetto. Il pacchetto usa solo la libreria standard, quindi non installa dipendenze esterne.

L'ultima riga rende il pacchetto importabile subito anche nelle celle Python: in Colab un'installazione con `pip install -e` diventa visibile al notebook solo dopo un riavvio del runtime.

In [ ]:
import os
import sys

if not os.path.exists("/content/kb-parser"):
    !git clone -q https://github.com/massimiliano-cantore/kb-parser.git /content/kb-parser

%cd /content/kb-parser
!pip install -q -e .

if "/content/kb-parser/src" not in sys.path:
    sys.path.insert(0, "/content/kb-parser/src")

# Parte A — Riga di comando

## A.1 L'archivio di prova

Diciotto file fittizi nei tre formati, distribuiti in sottocartelle, con alcuni casi limite costruiti apposta: file vuoto, caratteri speciali, codifiche Latin-1 e Windows-1252, Markdown senza titolo, file molto lungo, estensione non prevista, CSV malformato, file binario con estensione `.txt` e un file nascosto. La descrizione di ogni caso è nel README.

In [ ]:
!find sample_archive -type f | sort

## A.2 Costruzione dell'indice e riepilogo

Il comando `build` legge la cartella e le sottocartelle, applica a ogni file il lettore del suo formato e salva l'indice in `index.json`.

I file che non si possono elaborare vengono segnalati e saltati senza interrompere il processo. Al termine il programma stampa il riepilogo: documenti elaborati, documenti scartati e relative motivazioni.

In [ ]:
!python -m kb_parser build sample_archive --output index.json

## A.3 L'indice salvato in JSON

Ogni documento è rappresentato con la stessa struttura, qualunque sia il formato di origine: titolo, data (se presente), numero di parole, estratto, formato, percorso e contenuto normalizzato.

In [ ]:
!head -n 30 index.json

## A.4 Elenco dei documenti

Ordinamento predefinito per percorso; con `--sort` anche per titolo, data o lunghezza.

In [ ]:
!python -m kb_parser list --index index.json

In [ ]:
!python -m kb_parser list --sort date --index index.json

## A.5 Ricerca per parola chiave

La ricerca avviene nel titolo e nel contenuto, senza distinzione tra maiuscole e minuscole, e trova anche parti di parola. I risultati sono ordinati per rilevanza (prima i documenti con la parola nel titolo, poi per numero di occorrenze) e mostrano un frammento di testo con la parola tra parentesi quadre.

In [ ]:
!python -m kb_parser search "manutenzione" --index index.json

In [ ]:
!python -m kb_parser search "LABORATORIO" --index index.json

## A.6 Filtro per formato

In [ ]:
!python -m kb_parser filter csv --index index.json

In [ ]:
!python -m kb_parser filter md --index index.json

## A.7 Statistiche di sintesi

Documenti per tipo, documenti più lunghi, date e file scartati durante la costruzione.

In [ ]:
!python -m kb_parser stats --index index.json

## A.8 Scheda completa di un documento

Il verbale qui sotto è salvato in codifica Latin-1: le lettere accentate vengono lette correttamente.

In [ ]:
!python -m kb_parser show verbali/verbale_collegio_docenti_2024.txt --index index.json

## A.9 Ricaricamento dell'indice senza rifare il parsing

I comandi `list`, `search`, `filter`, `stats` e `show` leggono soltanto `index.json`. Per dimostrarlo, la cartella dei documenti viene rinominata temporaneamente: la ricerca funziona ugualmente, perché il programma non rilegge i file originali.

In [ ]:
!mv sample_archive sample_archive_hidden
!python -m kb_parser search "verbale" --index index.json
!mv sample_archive_hidden sample_archive

## A.10 Errori dell'utente

Un indice inesistente o una cartella sbagliata producono un messaggio chiaro, senza traceback, e un codice di uscita diverso da zero.

In [ ]:
!python -m kb_parser stats --index indice_inesistente.json
!python -m kb_parser build cartella_inesistente

# Parte B — Uso da Python

## B.1 Ricaricamento dell'indice

`KnowledgeBaseIndex.load()` ricostruisce l'indice dal file JSON: ogni scheda torna a essere un oggetto `Document`, senza leggere di nuovo i documenti originali.

In [ ]:
from kb_parser.index import KnowledgeBaseIndex

index = KnowledgeBaseIndex.load("index.json")
print(f"Documenti caricati da index.json: {len(index)}")
print(f"Indice creato il {index.created_at} dalla cartella '{index.source_folder}'")

document = index.get("procedure/accoglienza_nuovi_docenti.md")
print()
print(type(document).__name__)
print("title:      ", document.title)
print("date:       ", document.date)
print("word_count: ", document.word_count)
print("file_format:", document.file_format)
print("path:       ", document.path)
print("excerpt:    ", document.excerpt)

## B.2 Ricerca, filtro e statistiche

In [ ]:
for result in index.search("filtro"):
    print(f"{result.document.title}  (nel titolo: {result.in_title}, occorrenze: {result.occurrences})")
    print(f"    {result.snippet}")

print()
print("Documenti CSV:", [doc.path for doc in index.filter_by_format("csv")])

stats = index.statistics(top=3)
print()
print("Documenti per formato:", stats.documents_by_format)
print("I tre più lunghi:     ", [(doc.path, doc.word_count) for doc in stats.longest])
print("Scartati alla build:  ", stats.skipped_by_reason)

## B.3 Estendere il programma con un nuovo formato

Ogni formato è una classe derivata da `BaseReader`, che contiene i passi comuni (lettura, codifica, controlli, data, estratto, conteggio delle parole). Una nuova classe deve solo dichiarare le estensioni e implementare `parse()`.

Qui si aggiunge il supporto ai file `.log` con una sola classe, senza modificare scanner, indice o interfaccia.

In [ ]:
from pathlib import Path

from kb_parser.readers import BaseReader, register_reader, supported_extensions
from kb_parser.scanner import ArchiveScanner


class LogReader(BaseReader):
    """Reads .log files: the first line is the title, the content is the full text."""

    extensions = (".log",)

    def parse(self, text: str, path: Path) -> tuple[str | None, str]:
        first_line = text.strip().split("\n")[0]
        return first_line, text


demo = Path("/content/demo_log")
demo.mkdir(exist_ok=True)
(demo / "server.log").write_text(
    "Log del server della piattaforma\n2026-01-10 08:00 avvio\n2026-01-10 08:05 backup completato\n",
    encoding="utf-8",
)

_, report = ArchiveScanner(demo).scan()
print("Prima della registrazione:")
print(report.summary())

if ".log" not in supported_extensions():
    register_reader(LogReader)

documents, report = ArchiveScanner(demo).scan()
print()
print("Dopo la registrazione:")
print(report.summary())
print(documents[0].title, "|", documents[0].date, "|", documents[0].word_count, "parole")

# Parte C — Prova di robustezza

Il parser deve reggere su archivi che non sono stati preparati per lui. La cella seguente crea al momento una cartella con file problematici diversi da quelli dell'archivio di prova, insieme ad alcuni file validi, e la elabora: ogni problema viene segnalato con il suo motivo e l'elaborazione prosegue sugli altri file.

In [ ]:
import random
from pathlib import Path

test = Path("/content/robustness_test")
(test / "sottocartella" / "più livelli").mkdir(parents=True, exist_ok=True)
rng = random.Random(0)

files = {
    # validi, ma insoliti
    "NOTE IN MAIUSCOLO.TXT": "Nota con estensione maiuscola e spazi nel nome.".encode("utf-8"),
    "unicode_windows.txt": "Nota salvata dal Blocco note di Windows in UTF-16".encode("utf-16"),
    "sottocartella/più livelli/annotazione è.md": "# Titolo con accenti\nCartelle annidate e nomi accentati.".encode("utf-8"),
    "solo_una_colonna.csv": "matricola\nM-001\nM-002\n".encode("utf-8"),
    # problematici
    "senza_estensione": b"Un file senza estensione",
    "documento.docx": b"PK\x03\x04 finto documento Word",
    "immagine_rinominata.txt": bytes(rng.randrange(256) for _ in range(2048)),
    "solo_spazi.md": b"   \n\t\n   ",
    "solo_simboli.md": b"---\n***\n___\n",
    "virgolette_aperte.csv": 'codice,nota\nA1,"nota senza chiusura\n'.encode("utf-8"),
    "solo_intestazione.csv": b"codice;descrizione\n",
    "colonne_sbagliate.csv": b"a;b;c\n1;2;3\n4;5\n",
}
for name, data in files.items():
    (test / name).write_bytes(data)

!python -m kb_parser build /content/robustness_test --output /content/robustness_index.json

In [ ]:
!python -m kb_parser list --index /content/robustness_index.json